In [0]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from utils.utils_merge_into_tables import upsert_data
from utils.utils_transform_data import cast_columns, standardize_column_names, standardize_string_values
from pyspark.sql import functions as sf 



In [0]:
catalog = dbutils.widgets.get("catalog")
schema_bronze = dbutils.widgets.get("schema_bronze")
schema_silver = dbutils.widgets.get("schema_silver")
table_bronze = dbutils.widgets.get("table_bronze")
table_silver = dbutils.widgets.get("table_silver")
primary_keys = dbutils.widgets.get("primary_key")

In [0]:
df_bronze = spark.read.table(f"{catalog}.{schema_bronze}.{table_bronze}")

## Transform Data


In [0]:
df_bronze = df_bronze.select(
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
)

data_type_mapping = {
    "customer_id": "string",
    "customer_unique_id": "string",
    "customer_zip_code_prefix": "string",
    "customer_city": "string",
    "customer_state": "string"
}

## Silver

In [0]:
df_silver = (
    df_bronze
    .transform(lambda df: cast_columns(df, data_type_mapping))
    .transform(standardize_column_names)
    .transform(standardize_string_values)
    .withColumn("silver_update_date", sf.current_timestamp())
)

In [0]:
upsert_data(df_silver, table_silver,primary_keys)